# Feature Engineering + Feature Selection (Combined, Generalized)

Based on your original workflow: missing-value indicator flags, target-guided categorical
encoding, rare-label grouping, skew correction, MinMax scaling, and Lasso-based feature
selection — generalized so it isn't tied to House Prices column names, and with the scaler
fit only on train and correctly reapplied to test (no re-fitting, no data leakage).

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

pd.pandas.set_option("display.max_columns", None)


## Config

Adjust these for your dataset.

In [9]:
TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"          # set to None if you don't have a separate test file
TARGET_COL = "SalePrice"

RARE_LABEL_THRESHOLD = 0.01      # categories appearing in <1% of rows get grouped as "Rare_var"
SKEW_THRESHOLD = 0.75            # abs(skew) above this triggers a log transform
LASSO_ALPHA = 0.0005             # higher alpha = fewer features selected

# Optional: temporal columns you want converted to "years since" (dataset-specific — leave
# empty list [] if not applicable, or fill in manually, e.g. ["YearBuilt", "YearRemodAdd"]).
# If set, TEMPORAL_REFERENCE_COL is subtracted from each (e.g. YrSold - YearBuilt = house age).
TEMPORAL_COLS = []
TEMPORAL_REFERENCE_COL = None


## Load training data

In [10]:
dataset = pd.read_csv('train.csv')
print(f"Loaded {dataset.shape[0]} rows, {dataset.shape[1]} columns")
dataset.head()


Loaded 1460 rows, 81 columns


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Drop identifier-like columns (universal — no hardcoded column name)

Any column where every value is unique is almost certainly an ID column with no predictive
value, regardless of its name. We keep it aside (not just drop it) so it can be reattached
to test predictions later for submission purposes.

In [11]:
id_like_cols = [
    col for col in dataset.columns
    if col != TARGET_COL and dataset[col].nunique(dropna=False) == len(dataset)
]
print(f"Identifier-like columns detected: {id_like_cols}")

# Keep the first one aside as the "row ID" for later (e.g. Kaggle submission format)
ID_COL = id_like_cols[0] if id_like_cols else None
if ID_COL:
    train_ids = dataset[ID_COL].copy()
    dataset = dataset.drop(columns=id_like_cols)


Identifier-like columns detected: ['Id']


## Add missing-value indicator columns

For every column with missing values, add a companion `<col>_nan` flag (1 if missing, 0
otherwise) before filling — this preserves the "was this missing" signal, which can itself
be predictive (e.g. missing PoolQC often means "no pool", not "unknown").

In [12]:
features_with_nan = [f for f in dataset.columns if dataset[f].isnull().sum() > 0]

for feature in features_with_nan:
    dataset[feature + "_nan"] = np.where(dataset[feature].isnull(), 1, 0)

print(f"Added {len(features_with_nan)} missing-value indicator columns")
features_with_nan


Added 19 missing-value indicator columns


['LotFrontage',
 'Alley',
 'MasVnrType',
 'MasVnrArea',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Electrical',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PoolQC',
 'Fence',
 'MiscFeature']

## Handle missing values — categorical features

In [13]:
features_nan_cat = [f for f in dataset.columns
                     if dataset[f].isnull().sum() > 0 and dataset[f].dtype == "O"]

for feature in features_nan_cat:
    pct = np.round(dataset[feature].isnull().mean(), 4)
    print(f"{feature} - {pct} missing")

def fill_categorical_nan(df, cols, fill_value="Missing"):
    df = df.copy()
    df[cols] = df[cols].fillna(fill_value)
    return df

dataset = fill_categorical_nan(dataset, features_nan_cat)
print("\nRemaining NaNs in these columns:", dataset[features_nan_cat].isnull().sum().sum())


Alley - 0.9377 missing
MasVnrType - 0.5973 missing
BsmtQual - 0.0253 missing
BsmtCond - 0.0253 missing
BsmtExposure - 0.026 missing
BsmtFinType1 - 0.0253 missing
BsmtFinType2 - 0.026 missing
Electrical - 0.0007 missing
FireplaceQu - 0.4726 missing
GarageType - 0.0555 missing
GarageFinish - 0.0555 missing
GarageQual - 0.0555 missing
GarageCond - 0.0555 missing
PoolQC - 0.9952 missing
Fence - 0.8075 missing
MiscFeature - 0.963 missing

Remaining NaNs in these columns: 0


## Handle missing values — numerical features

Medians are computed on train only and stored, so the exact same values fill test later (no leakage).

In [14]:
features_nan_num = [f for f in dataset.columns
                     if dataset[f].isnull().sum() > 0 and dataset[f].dtype != "O"]

for feature in features_nan_num:
    pct = np.round(dataset[feature].isnull().mean(), 4)
    print(f"{feature} - {pct} missing")

# Store medians computed on TRAIN ONLY, reused on test — never recompute medians on test data
train_medians = {f: dataset[f].median() for f in features_nan_num}

def fill_numeric_nan(df, medians):
    df = df.copy()
    for feature, median_value in medians.items():
        if feature in df.columns:
            df[feature] = df[feature].fillna(median_value)
    return df

dataset = fill_numeric_nan(dataset, train_medians)
print("\nRemaining NaNs in these columns:", dataset[features_nan_num].isnull().sum().sum())


LotFrontage - 0.1774 missing
MasVnrArea - 0.0055 missing
GarageYrBlt - 0.0555 missing

Remaining NaNs in these columns: 0


## Handle temporal columns (optional, dataset-specific)

Only runs if `TEMPORAL_COLS` and `TEMPORAL_REFERENCE_COL` are set in the config above —
converts e.g. `YearBuilt` into "years since built" relative to a reference column
(e.g. `YrSold`), which is usually more predictive than a raw year.

In [15]:
if TEMPORAL_COLS and TEMPORAL_REFERENCE_COL:
    for feature in TEMPORAL_COLS:
        dataset[feature] = dataset[TEMPORAL_REFERENCE_COL] - dataset[feature]
    print(f"Converted to relative years: {TEMPORAL_COLS}")
else:
    print("No temporal columns configured — skipping.")


No temporal columns configured — skipping.


## Handle skewed numeric features (auto-detected, not hardcoded)

Any numeric column (excluding the target and the nan-flag columns, which are already 0/1)
with `abs(skew) > SKEW_THRESHOLD` gets log-transformed. The target itself is also checked
and transformed separately, since it needs `np.expm1()` on predictions afterward.

In [16]:
from scipy import stats

nan_flag_cols = [f + "_nan" for f in features_with_nan]
numeric_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()
candidate_cols = [c for c in numeric_cols if c not in nan_flag_cols and c != TARGET_COL]

skewed = dataset[candidate_cols].apply(lambda x: stats.skew(x.dropna()))
skewed_features = skewed[abs(skewed) > SKEW_THRESHOLD].index.tolist()

# Only log-transform strictly positive columns (log of 0/negative is invalid)
skewed_features = [c for c in skewed_features if (dataset[c] > 0).all()]

for feature in skewed_features:
    dataset[feature] = np.log(dataset[feature])

print(f"Log-transformed {len(skewed_features)} skewed columns: {skewed_features}")

# Transform the target too, if skewed
target_is_log_transformed = False
if TARGET_COL in dataset.columns and (dataset[TARGET_COL] > 0).all():
    target_skew = stats.skew(dataset[TARGET_COL])
    if abs(target_skew) > SKEW_THRESHOLD:
        dataset[TARGET_COL] = np.log(dataset[TARGET_COL])
        target_is_log_transformed = True
        print(f"Target \'{TARGET_COL}\' log-transformed (skew was {target_skew:.3f})")


Log-transformed 5 skewed columns: ['MSSubClass', 'LotFrontage', 'LotArea', '1stFlrSF', 'GrLivArea']
Target 'SalePrice' log-transformed (skew was 1.881)


## Handle rare categorical labels

Categories making up less than `RARE_LABEL_THRESHOLD` of rows get grouped into `"Rare_var"`.
This reduces overfitting to categories with very few examples. The kept-category list is
stored per column so test data uses the exact same grouping.

In [17]:
categorical_features = [f for f in dataset.columns if dataset[f].dtype == "O"]

frequent_labels = {}  # stored per column, reused on test

for feature in categorical_features:
    freq = dataset[feature].value_counts() / len(dataset)
    kept = freq[freq > RARE_LABEL_THRESHOLD].index
    frequent_labels[feature] = kept
    dataset[feature] = np.where(dataset[feature].isin(kept), dataset[feature], "Rare_var")

print(f"Applied rare-label grouping to {len(categorical_features)} categorical columns")


Applied rare-label grouping to 43 categorical columns


## Target-guided ordinal encoding

Each categorical column's labels are ranked by mean target value and mapped to integers —
this often works better than plain one-hot for tree/linear models on ordinal-like categories.
The mapping is stored per column so test data uses the identical encoding (any unseen
category on test falls back to NaN, then gets median-filled afterward).

In [18]:
label_mappings = {}  # stored per column, reused on test

for feature in categorical_features:
    labels_ordered = dataset.groupby(feature)[TARGET_COL].mean().sort_values().index
    mapping = {k: i for i, k in enumerate(labels_ordered, 0)}
    label_mappings[feature] = mapping
    dataset[feature] = dataset[feature].map(mapping)

print(f"Encoded {len(categorical_features)} categorical columns using target-guided ordinal mapping")
dataset.head()


Encoded 43 categorical columns using target-guided ordinal mapping


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,LotFrontage_nan,Alley_nan,MasVnrType_nan,MasVnrArea_nan,BsmtQual_nan,BsmtCond_nan,BsmtExposure_nan,BsmtFinType1_nan,BsmtFinType2_nan,Electrical_nan,FireplaceQu_nan,GarageType_nan,GarageYrBlt_nan,GarageFinish_nan,GarageQual_nan,GarageCond_nan,PoolQC_nan,Fence_nan,MiscFeature_nan
0,4.094345,3,4.174387,9.041922,1,2,0,1,1,0,0,14,2,1,3,5,7,5,2003,2003,0,0,10,10,2,196.0,2,3,4,3,3,1,6,706,5,0,150,856,2,4,1,3,6.752270,854,0,7.444249,1,0,2,1,3,1,2,8,4,0,1,4,2003.0,2,2,548,2,3,2,0,61,0,0,0,0,0,4,2,0,2,2008,2,3,12.247694,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,1
1,2.995732,3,4.382027,9.169518,1,2,0,1,1,2,0,11,1,1,3,3,6,8,1976,1976,0,0,4,3,1,0.0,1,3,2,3,3,4,4,978,5,0,284,1262,2,4,1,3,7.140453,0,0,7.140453,0,1,2,0,3,1,1,6,4,1,3,4,1976.0,2,2,460,2,3,2,298,0,0,0,0,0,0,4,2,0,5,2007,2,3,12.109011,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1
2,4.094345,3,4.219508,9.328123,1,2,1,1,1,0,0,14,2,1,3,5,7,5,2001,2002,0,0,10,10,2,162.0,2,3,4,3,3,2,6,486,5,0,434,920,2,4,1,3,6.824374,866,0,7.487734,1,0,2,1,3,1,2,6,4,1,3,4,2001.0,2,2,608,2,3,2,0,42,0,0,0,0,0,4,2,0,9,2008,2,3,12.317167,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1
3,4.248495,3,4.094345,9.164296,1,2,1,1,1,1,0,16,2,1,3,5,7,5,1915,1970,0,0,2,4,1,0.0,1,3,1,2,4,1,4,216,5,0,540,756,2,3,1,3,6.867974,756,0,7.448334,1,0,1,0,3,1,2,7,4,1,4,2,1998.0,1,3,642,2,3,2,0,35,272,0,0,0,0,4,2,0,2,2006,2,0,11.849398,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1
4,4.094345,3,4.430817,9.565214,1,2,1,1,1,2,0,22,2,1,3,5,8,5,2000,2000,0,0,10,10,2,350.0,2,3,4,3,3,3,6,655,5,0,490,1145,2,4,1,3,7.043160,1053,0,7.695303,1,0,2,1,4,1,2,9,4,1,3,4,2000.0,2,3,836,2,3,2,192,84,0,0,0,0,0,4,2,0,12,2008,2,3,12.429216,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1


## Feature scaling

MinMaxScaler is **fit on train only** and stored — the same fitted scaler transforms test
later (never re-fit on test data, which would leak test distribution info into the pipeline).

In [19]:
scalable_features = [f for f in dataset.columns if f != TARGET_COL]

scaler = MinMaxScaler()
scaler.fit(dataset[scalable_features])

dataset_scaled = pd.DataFrame(
    scaler.transform(dataset[scalable_features]),
    columns=scalable_features,
    index=dataset.index
)

# Reattach target (and ID, if present) for a complete, clean dataframe
data = pd.concat([dataset[[TARGET_COL]].reset_index(drop=True), dataset_scaled.reset_index(drop=True)], axis=1)
if ID_COL:
    data.insert(0, ID_COL, train_ids.reset_index(drop=True))

data.head()


,Id,SalePrice,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,LotFrontage_nan,Alley_nan,MasVnrType_nan,MasVnrArea_nan,BsmtQual_nan,BsmtCond_nan,BsmtExposure_nan,BsmtFinType1_nan,BsmtFinType2_nan,Electrical_nan,FireplaceQu_nan,GarageType_nan,GarageYrBlt_nan,GarageFinish_nan,GarageQual_nan,GarageCond_nan,PoolQC_nan,Fence_nan,MiscFeature_nan
0,1,12.247694,0.487992,0.75,0.418208,0.366344,1.0,1.0,0.000000,0.333333,1.0,0.00,0.0,0.636364,0.4,1.0,0.75,1.0,0.666667,0.500,0.949275,0.883333,0.0,0.0,1.0,1.0,0.666667,0.12250,0.666667,1.0,1.00,0.75,0.75,0.25,1.000000,0.125089,0.833333,0.0,0.064212,0.140098,1.0,1.00,1.0,1.0,0.356155,0.413559,0.0,0.577712,0.333333,0.0,0.666667,0.5,0.375,0.333333,0.666667,0.500000,1.0,0.000000,0.2,0.8,0.936364,0.666667,0.50,0.386460,0.666667,1.0,1.0,0.000000,0.111517,0.000000,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.090909,0.50,0.666667,0.75,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,2,12.109011,0.000000,0.75,0.495064,0.391317,1.0,1.0,0.000000,0.333333,1.0,0.50,0.0,0.500000,0.2,1.0,0.75,0.6,0.555556,0.875,0.753623,0.433333,0.0,0.0,0.4,0.3,0.333333,0.00000,0.333333,1.0,0.50,0.75,0.75,1.00,0.666667,0.173281,0.833333,0.0,0.121575,0.206547,1.0,1.00,1.0,1.0,0.503056,0.000000,0.0,0.470245,0.000000,0.5,0.666667,0.0,0.375,0.333333,0.333333,0.333333,1.0,0.333333,0.6,0.8,0.690909,0.666667,0.50,0.324401,0.666667,1.0,1.0,0.347725,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.363636,0.25,0.666667,0.75,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,3,12.317167,0.487992,0.75,0.434909,0.422359,1.0,1.0,0.333333,0.333333,1.0,0.00,0.0,0.636364,0.4,1.0,0.75,1.0,0.666667,0.500,0.934783,0.866667,0.0,0.0,1.0,1.0,0.666667,0.10125,0.666667,1.0,1.00,0.75,0.75,0.50,1.000000,0.086109,0.833333,0.0,0.185788,0.150573,1.0,1.00,1.0,1.0,0.383441,0.419370,0.0,0.593095,0.333333,0.0,0.666667,0.5,0.375,0.333333,0.666667,0.333333,1.0,0.333333,0.6,0.8,0.918182,0.666667,0.50,0.428773,0.666667,1.0,1.0,0.000000,0.076782,0.000000,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.727273,0.50,0.666667,0.75,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
3,4,11.849398,0.556464,0.75,0.388581,0.390295,1.0,1.0,0.333333,0.333333,1.0,0.25,0.0,0.727273,0.4,1.0,0.75,1.0,0.666667,0.500,0.311594,0.333333,0.0,0.0,0.2,0.4,0.333333,0.00000,0.333333,1.0,0.25,0.50,1.00,0.25,0.666667,0.038271,0.833333,0.0,0.231164,0.123732,1.0,0.75,1.0,1.0,0.399941,0.366102,0.0,0.579157,0.333333,0.0,0.333333,0.0,0.375,0.333333,0.666667,0.416667,1.0,0.333333,0.8,0.4,0.890909,0.333333,0.75,0.452750,0.666667,1.0,1.0,0.000000,0.063985,0.492754,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.090909,0.00,0.666667,0.00,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,5,12.429216,0.487992,0.75,0.513123,0.468761,1.0,1.0,0.333333,0.333333,1.0,0.50,0.0,1.000000,0.4,1.0,0.75,1.0,0.777778,0.500,0.927536,0.833333,0.0,0.0,1.0,1.0,0.666667,0.21875,0.666667,1.0,1.00,0.75,0.75,0.75,1.000000,0.116052,0.833333,0.0,0.209760,0.187398,1.0,1.00,1.0,1.0,0.466237,0.509927,0.0,0.666523,0.333333,0.0,0.666667,0.5,0.500,0.333333,0.666667,0.583333,1.0,0.333333,0.6,0.8,0.909091,0.666667,0.75,0.589563,0.666667,1.0,1.0,0.224037,0.153565,0.000000,0.0,0.0,0.0,0.0,1.0,1.

## Save engineered (pre-selection) training data

In [20]:
data.to_csv("train_engineered.csv", index=False)
print(f"Saved train_engineered.csv — shape {data.shape}")


Saved train_engineered.csv — shape (1460, 100)


## Feature selection (Lasso + SelectFromModel)

Fits a Lasso regression and keeps only features with non-zero coefficients. `alpha` controls
strictness — higher alpha keeps fewer features. Works for regression targets; for a
classification target, swap `Lasso` for `LogisticRegression(penalty='l1', solver='liblinear')`.

In [21]:
feature_cols = [c for c in data.columns if c not in [TARGET_COL, ID_COL]]
X_train = data[feature_cols]
y_train = data[TARGET_COL]

feature_sel_model = SelectFromModel(Lasso(alpha=LASSO_ALPHA, random_state=0))
feature_sel_model.fit(X_train, y_train)

selected_feat = X_train.columns[feature_sel_model.get_support()]

print(f"Total features: {X_train.shape[1]}")
print(f"Selected features: {len(selected_feat)}")
print(f"Features shrunk to zero: {np.sum(feature_sel_model.estimator_.coef_ == 0)}")
print("\nSelected features:")
list(selected_feat)


Total features: 98
Selected features: 50
Features shrunk to zero: 48

Selected features:


['MSSubClass',
 'MSZoning',
 'LotArea',
 'Alley',
 'LandContour',
 'LotConfig',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'HouseStyle',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'MasVnrType',
 'MasVnrArea',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtExposure',
 'BsmtFinType2',
 'BsmtUnfSF',
 'HeatingQC',
 'CentralAir',
 '1stFlrSF',
 '2ndFlrSF',
 'GrLivArea',
 'BsmtFullBath',
 'FullBath',
 'HalfBath',
 'KitchenQual',
 'Functional',
 'Fireplaces',
 'FireplaceQu',
 'GarageFinish',
 'GarageCars',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'WoodDeckSF',
 'ScreenPorch',
 'Fence',
 'YrSold',
 'SaleCondition',
 'FireplaceQu_nan',
 'MiscFeature_nan']

In [22]:
X_train_selected = X_train[selected_feat]

final_train = pd.concat([X_train_selected.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
if ID_COL:
    final_train.insert(0, ID_COL, data[ID_COL].reset_index(drop=True))

final_train.to_csv("X_train.csv", index=False)
print(f"Saved X_train.csv — shape {final_train.shape}")
final_train.head()


Saved X_train.csv — shape (1460, 52)


,Id,MSSubClass,MSZoning,LotArea,Alley,LandContour,LotConfig,Neighborhood,Condition1,Condition2,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtExposure,BsmtFinType2,BsmtUnfSF,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,GrLivArea,BsmtFullBath,FullBath,HalfBath,KitchenQual,Functional,Fireplaces,FireplaceQu,GarageFinish,GarageCars,GarageQual,GarageCond,PavedDrive,WoodDeckSF,ScreenPorch,Fence,YrSold,SaleCondition,FireplaceQu_nan,MiscFeature_nan,SalePrice
0,1,0.487992,0.75,0.366344,1.0,0.333333,0.00,0.636364,0.4,1.0,1.0,0.666667,0.500,0.949275,0.883333,0.0,0.0,1.0,0.666667,0.12250,0.666667,1.0,1.00,0.75,0.25,0.833333,0.064212,1.00,1.0,0.356155,0.413559,0.577712,0.333333,0.666667,0.5,0.666667,1.0,0.000000,0.2,0.666667,0.50,0.666667,1.0,1.0,0.000000,0.0,1.0,0.50,0.75,1.0,1.0,12.247694
1,2,0.000000,0.75,0.391317,1.0,0.333333,0.50,0.500000,0.2,1.0,0.6,0.555556,0.875,0.753623,0.433333,0.0,0.0,0.4,0.333333,0.00000,0.333333,1.0,0.50,0.75,1.00,0.833333,0.121575,1.00,1.0,0.503056,0.000000,0.470245,0.000000,0.666667,0.0,0.333333,1.0,0.333333,0.6,0.666667,0.50,0.666667,1.0,1.0,0.347725,0.0,1.0,0.25,0.75,0.0,1.0,12.109011
2,3,0.487992,0.75,0.422359,1.0,0.333333,0.00,0.636364,0.4,1.0,1.0,0.666667,0.500,0.934783,0.866667,0.0,0.0,1.0,0.666667,0.10125,0.666667,1.0,1.00,0.75,0.50,0.833333,0.185788,1.00,1.0,0.383441,0.419370,0.593095,0.333333,0.666667,0.5,0.666667,1.0,0.333333,0.6,0.666667,0.50,0.666667,1.0,1.0,0.000000,0.0,1.0,0.50,0.75,0.0,1.0,12.317167
3,4,0.556464,0.75,0.390295,1.0,0.333333,0.25,0.727273,0.4,1.0,1.0,0.666667,0.500,0.311594,0.333333,0.0,0.0,0.2,0.333333,0.00000,0.333333,1.0,0.25,0.50,0.25,0.833333,0.231164,0.75,1.0,0.399941,0.366102,0.579157,0.333333,0.333333,0.0,0.666667,1.0,0.333333,0.8,0.333333,0.75,0.666667,1.0,1.0,0.000000,0.0,1.0,0.00,0.00,0.0,1.0,11.849398
4,5,0.487992,0.75,0.468761,1.0,0.333333,0.50,1.000000,0.4,1.0,1.0,0.777778,0.500,0.927536,0.833333,0.0,0.0,1.0,0.666667,0.21875,0.666667,1.0,1.00,0.75,0.75,0.833333,0.209760,1.00,1.0,0.466237,0.509927,0.666523,0.333333,0.666667,0.5,0.666667,1.0,0.333333,0.6,0.666667,0.75,0.666667,1.0,1.0,0.224037,0.0,1.0,0.50,0.75,0.0,1.0,12.429216


## Apply the identical pipeline to test data

Uses only parameters already fit on train (medians, rare-label lists, encoding maps, scaler)
— nothing is re-fit here, which is what prevents train/test leakage.

In [23]:
if TEST_FILE:
    test = pd.read_csv(TEST_FILE)

    test_ids = test[ID_COL].copy() if ID_COL and ID_COL in test.columns else None
    if ID_COL and ID_COL in test.columns:
        test = test.drop(columns=[ID_COL])

    # Same missing-value indicator flags
    for feature in features_with_nan:
        if feature in test.columns:
            test[feature + "_nan"] = np.where(test[feature].isnull(), 1, 0)

    # Same categorical missing-value handling
    test = fill_categorical_nan(test, [f for f in features_nan_cat if f in test.columns])

    # Same numeric missing-value handling (using TRAIN medians)
    test = fill_numeric_nan(test, train_medians)

    # Same temporal transform
    if TEMPORAL_COLS and TEMPORAL_REFERENCE_COL:
        for feature in TEMPORAL_COLS:
            if feature in test.columns:
                test[feature] = test[TEMPORAL_REFERENCE_COL] - test[feature]

    # Same skew log-transform (never touches the target — test has no target column)
    for feature in skewed_features:
        if feature in test.columns and (test[feature] > 0).all():
            test[feature] = np.log(test[feature])

    # Same rare-label grouping (using TRAIN-derived frequent label lists)
    for feature in categorical_features:
        if feature in test.columns:
            kept = frequent_labels[feature]
            test[feature] = np.where(test[feature].isin(kept), test[feature], "Rare_var")

    # Same target-guided encoding (using TRAIN-derived mappings)
    for feature in categorical_features:
        if feature in test.columns:
            test[feature] = test[feature].map(label_mappings[feature])
            # Unseen categories become NaN after mapping — fill with the median of the
            # already-encoded train column as a safe fallback
            if test[feature].isnull().sum() > 0:
                test[feature] = test[feature].fillna(dataset[feature].median())

    # Same scaler (fit on train only — transform only, no fit here)
    test_scaled = pd.DataFrame(
        scaler.transform(test[scalable_features]),
        columns=scalable_features,
        index=test.index
    )

    # Keep only the features selected from train
    test_final = test_scaled[selected_feat]
    if ID_COL and test_ids is not None:
        test_final.insert(0, ID_COL, test_ids.reset_index(drop=True))

    test_final.to_csv("X_test.csv", index=False)
    print(f"Saved X_test.csv — shape {test_final.shape}")
    test_final.head()
else:
    print("TEST_FILE not set — skipping test transformation.")


Saved X_test.csv — shape (1459, 51)


## Notes

- If `target_is_log_transformed` is `True`, remember to apply `np.expm1()` to your model's
  predictions to convert back to the original scale before submitting/reporting them.
- `Lasso`-based selection assumes a **regression** target. For classification, swap in
  `from sklearn.linear_model import LogisticRegression` with `penalty="l1", solver="liblinear"`.
- `RARE_LABEL_THRESHOLD`, `SKEW_THRESHOLD`, and `LASSO_ALPHA` are reasonable defaults, not
  guaranteed-optimal — worth tuning per dataset.